# Notebook 04 — Baseline Model Training & Validation

**Member 3 deliverable — baseline matching module**

---

## Role boundary

| Module | Owner |
|---|---|
| `src/preprocess.py` | **Member 2** — do not modify |
| `output/candidate_pairs.tsv` | **Member 4** — do not regenerate here |
| `src/features.py`, `src/train.py`, `src/predict.py` | **Member 3** |
| Advanced model experiments | **Team lead** |

This notebook demonstrates:
1. Pair construction from M4 candidates + M2-normalized source records
2. 16 baseline feature extraction
3. Entity-level train/validation split (80/20)
4. StandardScaler + LogisticRegression training
5. Threshold sweep using the **official entity-level macro F0.5**
6. Error analysis on the validation set

---

## 1. Setup

In [ ]:
# ── Mount Drive and discover data directory (Colab) ──────────────────────────
import sys, os
from pathlib import Path

# Allow import of src/ modules
REPO_ROOT = Path().resolve().parent  # adjust if running from a different working dir
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# M2 normalization (Member 2 — never reimplemented here)
from preprocess import normalize_name, normalize_address

# M3 baseline features
from features import build_feature_matrix, FEATURE_NAMES, extract_features

# M3 training helpers
from train import build_truth_map, entity_split, threshold_sweep, macro_f05, entity_f05

print('Imports OK')
print('Features:', FEATURE_NAMES)

In [ ]:
# ── Load data ─────────────────────────────────────────────────────────────────
# Adjust DATA_DIR to your Colab/Drive path.
# If using the rglob auto-discovery from notebooks/01_eda.ipynb, set
# DATA_DIR = dataset_dir  (the variable produced by that cell).

DATA_DIR = Path('/content/drive/MyDrive/Amazon ML Challenge 2026/01_Dataset/ raw/train')
CAND_PATH = REPO_ROOT / 'output' / 'candidate_pairs.tsv'   # M4 artifact

s1 = pd.read_csv(DATA_DIR / 'train_source1.tsv', sep='\t')
s2 = pd.read_csv(DATA_DIR / 'train_source2.tsv', sep='\t')
s3 = pd.read_csv(DATA_DIR / 'train_source3.tsv', sep='\t')
gt = pd.read_csv(DATA_DIR / 'train_ground_truth.tsv', sep='\t')

# Apply M2 normalization — imported directly, not reimplemented
for df in (s1, s2, s3):
    df['business_name_norm']    = df['business_name'].apply(normalize_name)
    df['business_address_norm'] = df['business_address'].apply(normalize_address)

print(f'S1={len(s1):,}  S2={len(s2):,}  S3={len(s3):,}  GT={len(gt):,}')

## 2. Candidate Pair Construction

In [ ]:
# ── Load M4 candidate pairs ───────────────────────────────────────────────────
# candidate_pairs.tsv is produced by Member 4.
# Do NOT regenerate candidate pairs here.
candidates = pd.read_csv(CAND_PATH, sep='\t')
print(f'Candidate rows: {len(candidates):,}')
display(candidates.head())

In [ ]:
from train import build_pair_rows

truth_map = build_truth_map(gt)
pair_df = build_pair_rows(candidates, s1, s2, s3, truth_map)

print(f'Total pairs : {len(pair_df):,}')
print(f'Positives   : {pair_df["label"].sum():,}  ({pair_df["label"].mean()*100:.2f}%)')
print(f'Negatives   : {(pair_df["label"]==0).sum():,}')
display(pair_df.head())

## 3. Feature Extraction

In [ ]:
X_all = build_feature_matrix(pair_df)
y_all = pair_df['label']

print('Feature matrix shape:', X_all.shape)
print('Features:', FEATURE_NAMES)

# Feature statistics by class
feat_stats = X_all.copy()
feat_stats['label'] = y_all.values
display(feat_stats.groupby('label')[FEATURE_NAMES].mean().T.rename(columns={0:'mean_neg', 1:'mean_pos'}))

## 4. Entity-Level Train / Validation Split

Split is by **unique S1 entity** (never row-level) to prevent leakage.

In [ ]:
train_df, val_df = entity_split(pair_df, train_frac=0.80, random_state=42)

train_s1 = set(train_df['source1_entity_id'].unique())
val_s1   = set(val_df['source1_entity_id'].unique())

print(f'Train — S1 entities: {len(train_s1):,}   pairs: {len(train_df):,}')
print(f'Val   — S1 entities: {len(val_s1):,}    pairs: {len(val_df):,}')
print(f'ID overlap (must be 0): {len(train_s1 & val_s1)}')

X_train = build_feature_matrix(train_df).values
y_train = train_df['label'].values
X_val   = build_feature_matrix(val_df).values
y_val   = val_df['label'].values

## 5. Baseline Model Training

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_val_sc   = scaler.transform(X_val)

model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train_sc, y_train)

val_proba = model.predict_proba(X_val_sc)[:, 1]

# Coefficient table
coef_df = pd.DataFrame({'feature': FEATURE_NAMES, 'coefficient': model.coef_[0]})
coef_df = coef_df.sort_values('coefficient', key=abs, ascending=False)
print('Feature coefficients (by |magnitude|):')
display(coef_df)

## 6. Threshold Sweep — Official Entity-Level Macro F0.5

For each threshold t:
- Each S1 entity gets its own precision, recall, and F0.5
- Final score = **mean(entity F0.5)**
- Zero-match entity predicted as empty → F0.5 = 1.0
- Zero-match entity predicted as non-empty → F0.5 = 0.0

In [ ]:
sweep_df = threshold_sweep(val_df, val_proba, truth_map)
display(sweep_df)

best_row = sweep_df.loc[sweep_df['f0_5'].idxmax()]
print(f"\nBest threshold : {best_row['threshold']:.2f}")
print(f"Precision      : {best_row['precision']:.4f}")
print(f"Recall         : {best_row['recall']:.4f}")
print(f"F0.5           : {best_row['f0_5']:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(sweep_df['threshold'], sweep_df['f0_5'],     'b-o', label='F0.5', linewidth=2)
ax.plot(sweep_df['threshold'], sweep_df['precision'], 'g--', label='Precision')
ax.plot(sweep_df['threshold'], sweep_df['recall'],    'r--', label='Recall')
ax.axvline(best_row['threshold'], color='navy', linestyle=':', label=f"Best t={best_row['threshold']:.2f}")
ax.set_xlabel('Decision Threshold')
ax.set_ylabel('Score')
ax.set_title('Entity-Level Macro F0.5 — Threshold Sweep')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Error Analysis

In [ ]:
threshold = float(best_row['threshold'])
val_pred = val_proba >= threshold

val_df = val_df.copy()
val_df['score'] = val_proba
val_df['predicted'] = val_pred.astype(int)

fp = val_df[(val_df['label'] == 0) & (val_df['predicted'] == 1)]
fn = val_df[(val_df['label'] == 1) & (val_df['predicted'] == 0)]

print(f'False positives : {len(fp):,}')
print(f'False negatives : {len(fn):,}')

print('\n--- False positive feature means ---')
display(fp[FEATURE_NAMES].mean().to_frame('FP mean').T)

print('--- False negative feature means ---')
display(fn[FEATURE_NAMES].mean().to_frame('FN mean').T)

In [ ]:
print('False positive examples (predicted match, wrong):')
display(fp[['source1_entity_id', 'cand_entity_id', 'score',
            'name_jaccard', 'address_jaccard', 'address_missing']].head(10))

In [ ]:
print('False negative examples (missed match):')
display(fn[['source1_entity_id', 'cand_entity_id', 'score',
            'name_jaccard', 'name_levenshtein_ratio', 'address_jaccard']].head(10))

## 8. Save Artifacts

In [ ]:
import pickle, json

models_dir = REPO_ROOT / 'models'
output_dir = REPO_ROOT / 'output'
models_dir.mkdir(exist_ok=True)
output_dir.mkdir(exist_ok=True)

# Save model
with open(models_dir / 'matcher.pkl', 'wb') as f:
    pickle.dump((scaler, model), f)

# Save config
config = {
    'model_type':            'StandardScaler + LogisticRegression',
    'feature_names':         FEATURE_NAMES,
    'threshold':             float(best_row['threshold']),
    'random_seed':           42,
    'train_s1_count':        len(train_s1),
    'validation_s1_count':   len(val_s1),
    'train_pair_count':      int(len(train_df)),
    'validation_pair_count': int(len(val_df)),
    'precision':             round(float(best_row['precision']), 6),
    'recall':                round(float(best_row['recall']), 6),
    'f0_5':                  round(float(best_row['f0_5']), 6),
    'baseline_version':      '1.0',
    'metric':                'entity-level macro F0.5 (competition official)',
}

with open(models_dir / 'model_config.json', 'w') as f:
    json.dump(config, f, indent=2)

with open(output_dir / 'baseline_training_results.json', 'w') as f:
    json.dump(config, f, indent=2)

sweep_df.to_csv(output_dir / 'baseline_threshold_results.tsv', sep='\t', index=False)

print('Saved:')
for p in [
    models_dir / 'matcher.pkl',
    models_dir / 'model_config.json',
    output_dir / 'baseline_training_results.json',
    output_dir / 'baseline_threshold_results.tsv',
]:
    print(f'  {p}')
print('\nConfig:')
print(json.dumps(config, indent=2))

---

## Summary

| Item | Value |
|---|---|
| Model | StandardScaler + LogisticRegression |
| Features | 16 (name ×6, address ×7, other ×3) |
| Normalization | M2 `src/preprocess.py` |
| Candidates | M4 `output/candidate_pairs.tsv` |
| Split | 80/20 by S1 entity, random_state=42 |
| Metric | Entity-level macro F0.5 |
| Selected threshold | See sweep output above |
| Validation F0.5 | See sweep output above |

**This is the baseline reference point. All future model experiments by the team lead must beat this F0.5 score to justify increased complexity.**